# Final prediction for all four tasks

This notebook assembles the final classification submission in the issued template order and validates the Task 4 retrieval artefacts. Tasks 1--3 own their model-specific inference; this notebook consumes their prediction CSVs, rejects stale or misaligned rows, and writes one combined file only when every required column is complete. Task 2 is intentionally isolated in one configuration entry so its final artefact can be swapped without changing the merge logic.


## 1. Configuration

Set `WRITE_FINAL = True` only for the last submission run. Until then, the notebook performs a preflight and writes a clearly named draft only when all classification predictions are available.


In [ ]:
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'pyproject.toml').is_file(), 'Run this notebook from the repository root or notebooks/'

GROUP_ID = 'SG_G3'
WRITE_FINAL = False
RUN_TASK4_TEST_RETRIEVAL = False
TASK4_TOP_K = 10

# Only this mapping should need changing when Task 2 publishes its final CSV.
PREDICTION_SOURCES = {
    'articleType': [
        PROJECT_ROOT / 'predictions/task1/task1_predictions.csv',
        PROJECT_ROOT / 'predictions/task1_predictions.csv',
    ],
    'season': [
        PROJECT_ROOT / 'predictions/task2/task2_predictions.csv',
        PROJECT_ROOT / 'predictions/task2_predictions.csv',
    ],
    'gender_usage': [
        PROJECT_ROOT / 'predictions/task3/task3_gender_usage_nguyen.csv',
        PROJECT_ROOT / 'predictions/task3_gender_usage_nguyen.csv',
    ],
}

FINAL_DIR = PROJECT_ROOT / 'predictions/final'
FINAL_PATH = FINAL_DIR / f'COSC2753_A2_{GROUP_ID}.csv'
DRAFT_PATH = FINAL_DIR / f'DRAFT_COSC2753_A2_{GROUP_ID}.csv'
print('Project:', PROJECT_ROOT)
print('Final output:', FINAL_PATH)


## 2. Resolve the issued template and test images

The template is the authority for row count, ID values, row order, and column order. The resolver supports the repository layout and the two local staging layouts used by the task notebooks.


In [ ]:
EXPECTED_COLUMNS = ['id', 'gender', 'articleType', 'season', 'usage']
TEMPLATE_CANDIDATES = [
    PROJECT_ROOT / 'preprocessed_datasets/test/styles_prediction.csv',
    PROJECT_ROOT / 'datasets/test/styles_prediction.csv',
    Path('D:/ColabDataset/preprocessed_datasets/test/styles_prediction.csv'),
    Path('D:/ColabDataset/test/styles_prediction.csv'),
    # Safe preflight fallback: Task 1 preserved every issued template row and column.
    PROJECT_ROOT / 'predictions/task1/task1_predictions.csv',
]
TEST_IMAGE_DIR_CANDIDATES = [
    PROJECT_ROOT / 'preprocessed_datasets/test/images_test',
    PROJECT_ROOT / 'datasets/test/images_test',
    Path('D:/ColabDataset/preprocessed_datasets/test/images_test'),
    Path('D:/ColabDataset/test/images_test'),
]
TEMPLATE_PATH = next((p.resolve() for p in TEMPLATE_CANDIDATES if p.is_file()), None)
assert TEMPLATE_PATH is not None, 'No issued template or Task 1 template-preserving prediction CSV was found'
TEST_IMAGE_DIR = next((p.resolve() for p in TEST_IMAGE_DIR_CANDIDATES if p.is_dir()), None)
template = pd.read_csv(TEMPLATE_PATH, dtype={'id': 'int64'})
assert list(template.columns) == EXPECTED_COLUMNS, (template.columns.tolist(), EXPECTED_COLUMNS)
assert template['id'].is_unique and template['id'].notna().all()
print(f'Template: {TEMPLATE_PATH} ({len(template):,} rows)')
print(f'Test images: {TEST_IMAGE_DIR or "NOT STAGED (classification merge can still run)"}')


## 3. Load and align Task 1--3 predictions

Each source must contain `id` plus its owned output column(s). Rows are joined by ID and then restored to template order; positional concatenation is never used. Blank labels, duplicate IDs, missing IDs, extra IDs, and conflicting non-owned columns fail loudly.


In [ ]:
TASK_COLUMNS = {
    'articleType': ['articleType'],
    'season': ['season'],
    'gender_usage': ['gender', 'usage'],
}

def first_existing(candidates: list[Path]) -> Path | None:
    return next((p.resolve() for p in candidates if p.is_file()), None)

def load_task_predictions(path: Path, owned_columns: list[str], expected_ids: pd.Series) -> pd.DataFrame:
    frame = pd.read_csv(path, dtype={'id': 'int64'})
    required = ['id', *owned_columns]
    missing_columns = [column for column in required if column not in frame.columns]
    assert not missing_columns, f'{path}: missing columns {missing_columns}'
    assert frame['id'].notna().all() and frame['id'].is_unique, f'{path}: IDs must be unique and non-null'
    for column in owned_columns:
        values = frame[column].astype('string').str.strip()
        assert values.notna().all() and values.ne('').all(), f'{path}: blank values in {column}'
        frame[column] = values
    expected = set(expected_ids.astype(int))
    actual = set(frame['id'].astype(int))
    assert actual == expected, (
        f'{path}: ID mismatch; missing={sorted(expected - actual)[:10]}, '
        f'extra={sorted(actual - expected)[:10]}'
    )
    return frame[required]

submission = template.copy()
source_records = []
missing_tasks = []
for task_name, owned_columns in TASK_COLUMNS.items():
    source_path = first_existing(PREDICTION_SOURCES[task_name])
    if source_path is None:
        missing_tasks.append(task_name)
        source_records.append({'task': task_name, 'status': 'MISSING', 'path': None})
        continue
    task_frame = load_task_predictions(source_path, owned_columns, template['id'])
    aligned = template[['id']].merge(task_frame, on='id', how='left', validate='one_to_one')
    for column in owned_columns:
        submission[column] = aligned[column]
    source_records.append({'task': task_name, 'status': 'READY', 'path': str(source_path)})

source_status = pd.DataFrame(source_records)
display(source_status)
display(submission.head())


## 4. Validate classification labels and write the combined CSV

Known label vocabularies are derived from the cleaned training manifest when it is available. The final file is blocked until Task 1, Task 2, and Task 3 are all present.


In [ ]:
TRAIN_MANIFEST_CANDIDATES = [
    PROJECT_ROOT / 'preprocessed_datasets/train/styles_train.csv',
    PROJECT_ROOT / 'preprocessed_datasets/train_manifest.csv',
    PROJECT_ROOT / 'datasets/train/styles_train.csv',
    Path('D:/ColabDataset/preprocessed_datasets/train/styles_train.csv'),
]
train_manifest_path = first_existing(TRAIN_MANIFEST_CANDIDATES)
label_audit = []
if train_manifest_path is not None:
    train_manifest = pd.read_csv(train_manifest_path, low_memory=False)
    for column in EXPECTED_COLUMNS[1:]:
        if column not in submission or submission[column].isna().any() or column not in train_manifest:
            continue
        known = set(train_manifest[column].dropna().astype(str).str.strip())
        predicted = set(submission[column].astype(str).str.strip())
        unknown = sorted(predicted - known)
        label_audit.append({'column': column, 'predicted_classes': len(predicted), 'unknown': unknown})
        assert not unknown, f'{column}: labels absent from training vocabulary: {unknown}'
display(pd.DataFrame(label_audit))

FINAL_DIR.mkdir(parents=True, exist_ok=True)
classification_ready = not missing_tasks
if classification_ready:
    assert list(submission.columns) == EXPECTED_COLUMNS
    assert submission['id'].equals(template['id']), 'Template row order changed'
    assert submission[EXPECTED_COLUMNS[1:]].notna().all().all(), 'Final labels contain nulls'
    assert submission[EXPECTED_COLUMNS[1:]].apply(lambda s: s.astype(str).str.strip().ne('').all()).all()
    output_path = FINAL_PATH if WRITE_FINAL else DRAFT_PATH
    submission.to_csv(output_path, index=False, lineterminator='\n')
    print(f'Wrote {len(submission):,} rows -> {output_path}')
else:
    output_path = None
    print('Combined CSV not written. Missing task artefacts:', ', '.join(missing_tasks))


## 5. Task 4 retrieval artefact preflight

Task 4 is a retrieval system and therefore has no column in `styles_prediction.csv`. Its submission contract is the frozen ArcFace encoder plus a gallery embedding matrix whose rows align exactly with the gallery IDs. The optional next section can encode every test image and save its Top-K catalogue neighbours.


In [ ]:
TASK4_DIR = PROJECT_ROOT / 'artifacts/task4/models/arcface'
TASK4_FILES = {
    'checkpoint': TASK4_DIR / 'best.pt',
    'gallery_embeddings': TASK4_DIR / 'gallery_embeddings.npy',
    'gallery_ids': TASK4_DIR / 'gallery_ids.npy',
    'preprocessing': PROJECT_ROOT / 'artifacts/task4/configs/image_preprocessing.json',
}
task4_status = pd.DataFrame([
    {'artefact': name, 'ready': path.is_file(), 'path': str(path)}
    for name, path in TASK4_FILES.items()
])
display(task4_status)
task4_ready = bool(task4_status['ready'].all())
if task4_ready:
    gallery_ids = np.load(TASK4_FILES['gallery_ids'], mmap_mode='r')
    gallery_embeddings = np.load(TASK4_FILES['gallery_embeddings'], mmap_mode='r')
    assert gallery_ids.ndim == 1 and gallery_embeddings.ndim == 2
    assert len(gallery_ids) == len(gallery_embeddings), 'Task 4 gallery IDs/embeddings are misaligned'
    assert len(np.unique(gallery_ids)) == len(gallery_ids), 'Task 4 gallery IDs are not unique'
    norms = np.linalg.norm(np.asarray(gallery_embeddings[: min(2048, len(gallery_embeddings))]), axis=1)
    assert np.allclose(norms, 1.0, atol=2e-3), 'Task 4 gallery embeddings are not L2-normalised'
    print(f'Task 4 ready: {len(gallery_ids):,} gallery items, {gallery_embeddings.shape[1]} dimensions')
else:
    print('Task 4 retrieval export is blocked until every required artefact exists.')


## 6. Optional Task 4 Top-K predictions for all test queries

This section is off during ordinary preflight. When enabled, it restores the ArcFace encoder, applies the recorded letterbox and normalization, retrieves by cosine similarity, and saves one row per `(query, rank)` pair. It does not alter the classification submission.


In [ ]:
if RUN_TASK4_TEST_RETRIEVAL:
    assert task4_ready, 'Task 4 artefacts failed preflight'
    import torch
    from PIL import Image
    from torch.utils.data import DataLoader, Dataset
    from torchvision import transforms
    from src.task4.models import ResNet18Encoder

    with TASK4_FILES['preprocessing'].open(encoding='utf-8') as handle:
        prep = json.load(handle)
    size = tuple(prep['resize']['target_size'])
    mean = prep['normalization']['mean_rgb']
    std = prep['normalization']['std_rgb']
    pad = tuple(prep['resize']['padding_color_rgb'])

    class Letterbox:
        def __init__(self, target_size, fill):
            self.height, self.width = target_size
            self.fill = fill
        def __call__(self, image):
            image = image.convert('RGB')
            scale = min(self.width / image.width, self.height / image.height)
            resized = image.resize((max(1, round(image.width * scale)), max(1, round(image.height * scale))), Image.Resampling.BILINEAR)
            canvas = Image.new('RGB', (self.width, self.height), self.fill)
            canvas.paste(resized, ((self.width - resized.width) // 2, (self.height - resized.height) // 2))
            return canvas

    transform = transforms.Compose([Letterbox(size, pad), transforms.ToTensor(), transforms.Normalize(mean, std)])

    class TestImages(Dataset):
        def __init__(self, ids): self.ids = [int(i) for i in ids]
        def __len__(self): return len(self.ids)
        def __getitem__(self, index):
            image_id = self.ids[index]
            with Image.open(TEST_IMAGE_DIR / f'{image_id}.jpg') as image:
                tensor = transform(image)
            return image_id, tensor

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(TASK4_FILES['checkpoint'], map_location='cpu', weights_only=False)
    state = checkpoint.get('model_state_dict', checkpoint)
    encoder_state = {k: v for k, v in state.items() if not k.startswith('arcface_loss.')}
    model = ResNet18Encoder()
    model.load_state_dict(encoder_state, strict=True)
    model.to(device).eval()

    assert TEST_IMAGE_DIR is not None, 'Stage images_test before running Task 4 retrieval'
    missing_images = [int(i) for i in template['id'] if not (TEST_IMAGE_DIR / f'{i}.jpg').is_file()]
    assert not missing_images, f'Missing {len(missing_images)} test images; first IDs: {missing_images[:10]}'
    query_ids, query_vectors = [], []
    loader = DataLoader(TestImages(template['id']), batch_size=256, shuffle=False, num_workers=0, pin_memory=device.type == 'cuda')
    with torch.inference_mode():
        for ids, images in loader:
            query_ids.extend(ids.numpy().astype(int).tolist())
            query_vectors.append(model(images.to(device, non_blocking=True)).cpu().numpy())
    query_vectors = np.concatenate(query_vectors).astype('float32')
    gallery_matrix = np.asarray(gallery_embeddings, dtype='float32')

    rows = []
    block_size = 256
    k = min(TASK4_TOP_K, len(gallery_ids))
    for start in range(0, len(query_vectors), block_size):
        scores = query_vectors[start:start + block_size] @ gallery_matrix.T
        top = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]
        top_scores = np.take_along_axis(scores, top, axis=1)
        order = np.argsort(-top_scores, axis=1)
        top = np.take_along_axis(top, order, axis=1)
        top_scores = np.take_along_axis(top_scores, order, axis=1)
        for offset, query_id in enumerate(query_ids[start:start + block_size]):
            for rank, (gallery_index, score) in enumerate(zip(top[offset], top_scores[offset]), start=1):
                rows.append({'query_id': query_id, 'rank': rank, 'gallery_id': int(gallery_ids[gallery_index]), 'cosine_similarity': float(score)})
    retrieval = pd.DataFrame(rows)
    retrieval_path = FINAL_DIR / f'COSC2753_A2_{GROUP_ID}_task4_top{TASK4_TOP_K}.csv'
    retrieval.to_csv(retrieval_path, index=False, lineterminator='\n')
    print(f'Wrote {len(retrieval):,} retrieval rows -> {retrieval_path}')
    display(retrieval.head(2 * TASK4_TOP_K))
else:
    print('Task 4 test retrieval skipped. Set RUN_TASK4_TEST_RETRIEVAL = True for the final artefact run.')


## 7. Final manifest and submission gate

The manifest records the exact input files and hashes used by the final run. It is written only when the combined classification CSV exists; Task 4 readiness is recorded separately because retrieval is not part of the issued CSV schema.


In [ ]:
def sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

if output_path is not None:
    reloaded = pd.read_csv(output_path, dtype={'id': 'int64'})
    assert list(reloaded.columns) == EXPECTED_COLUMNS
    assert reloaded['id'].equals(template['id'])
    assert reloaded[EXPECTED_COLUMNS[1:]].notna().all().all()
    manifest = {
        'group_id': GROUP_ID,
        'classification_file': str(output_path.relative_to(PROJECT_ROOT)),
        'classification_sha256': sha256(output_path),
        'template': str(TEMPLATE_PATH),
        'template_sha256': sha256(TEMPLATE_PATH),
        'rows': len(reloaded),
        'columns': EXPECTED_COLUMNS,
        'sources': source_records,
        'task4_ready': task4_ready,
        'task4_files': {name: {'path': str(path), 'sha256': sha256(path) if path.is_file() else None} for name, path in TASK4_FILES.items()},
    }
    manifest_path = output_path.with_suffix('.manifest.json')
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print('Manifest:', manifest_path)
    print('Classification SHA-256:', manifest['classification_sha256'])
    if WRITE_FINAL:
        print('FINAL SUBMISSION READY')
    else:
        print('DRAFT ONLY: inspect results, then set WRITE_FINAL = True and Run All.')
else:
    print('NOT READY:', {'missing_classification_tasks': missing_tasks, 'task4_ready': task4_ready})
